# Arc-Eager Dependency Parser — from scratch

A hands-on companion to [`README.md`](README.md). We build a transition-based
**arc-eager** parser in plain Python (no libraries), then use a **static oracle**
to parse *"He sent her a letter."* and print the exact trace table from the notes.

**What you'll build**
1. A `Configuration` class holding the **stack**, **buffer**, and **arcs**, with the four
   transitions (`SHIFT`, `LEFT-ARC`, `RIGHT-ARC`, `REDUCE`) — each checking its precondition.
2. A **static oracle** that, given the gold tree, chooses the correct transition at each step.
3. A **runner** that applies the oracle, records the trace, and verifies the output equals
   the gold tree.
4. A **your-turn** exercise: parse a new sentence yourself.


## 1 · The sentence and its gold tree

Token index `0` is the artificial `ROOT`. The gold tree is stored as
`gold[dependent] = (head, label)` — every word (except ROOT) has exactly one head.


In [ ]:
# Tokens: index 0 is ROOT
tokens = ["ROOT", "He", "sent", "her", "a", "letter", "."]

# Gold dependency tree:  dependent -> (head, label)
gold = {
    2: (0, "root"),    # ROOT   -> sent   (root)
    1: (2, "nsubj"),   # sent   -> He     (nsubj)
    3: (2, "iobj"),    # sent   -> her    (iobj)
    5: (2, "dobj"),    # sent   -> letter (dobj)
    4: (5, "det"),     # letter -> a      (det)
    6: (2, "punct"),   # sent   -> .      (punct)
}

def word(i):
    "Readable token for an index."
    return tokens[i]

# Split gold into quick lookups the oracle will use
gold_head  = {dep: h  for dep, (h, l) in gold.items()}   # dependent -> head
gold_label = {dep: l  for dep, (h, l) in gold.items()}   # dependent -> label

print("Sentence:", " ".join(tokens[1:]))
for dep, (h, l) in gold.items():
    print(f"  {tokens[h]:>6} -> {tokens[dep]:<6} ({l})")

## 2 · The `Configuration` class

The state of the parser is `(stack, buffer, arcs)`. Each transition mutates this state and
**asserts its precondition** first — if a precondition fails, the move is illegal.

- `heads` is a helper dict (`dependent -> head`) so we can cheaply check *"does this word
  already have a head?"*, which two of the preconditions need.


In [ ]:
class Configuration:
    def __init__(self, tokens):
        self.tokens = tokens
        self.stack  = [0]                        # ROOT starts on the stack
        self.buffer = list(range(1, len(tokens)))# all real words, left to right
        self.arcs   = []                         # committed arcs: (head, label, dependent)
        self.heads  = {}                         # dependent -> head (for preconditions)

    # ---- the four arc-eager transitions ----
    def shift(self):
        assert self.buffer, "SHIFT precondition failed: buffer is empty"
        self.stack.append(self.buffer.pop(0))

    def left_arc(self, label):
        s, b = self.stack[-1], self.buffer[0]
        assert s != 0,               "LEFT-ARC precondition failed: stack top is ROOT"
        assert s not in self.heads,  "LEFT-ARC precondition failed: stack top already has a head"
        self.arcs.append((b, label, s))          # b is head of s
        self.heads[s] = b
        self.stack.pop()                         # s is done, remove it

    def right_arc(self, label):
        s, b = self.stack[-1], self.buffer[0]
        self.arcs.append((s, label, b))          # s is head of b
        self.heads[b] = s
        self.stack.append(self.buffer.pop(0))    # push b (it may still take children)

    def reduce(self):
        s = self.stack[-1]
        assert s in self.heads, "REDUCE precondition failed: stack top has no head yet"
        self.stack.pop()

    # ---- helpers ----
    def is_terminal(self):
        return len(self.buffer) == 0

    def snapshot(self):
        "Readable (stack, buffer) for the trace table."
        st = "[" + ", ".join(self.tokens[i] for i in self.stack) + "]"
        bf = "[" + ", ".join(self.tokens[i] for i in self.buffer) + "]"
        return st, bf

print("Configuration class defined.")

## 3 · The static oracle

The oracle knows the gold tree and returns the **correct** transition for the current
configuration. It checks the rules **in order** (first match wins) — exactly the decision
rule from the notes:

1. `LEFT-ARC`  — `b` is the gold head of `s` (and `s` has no head, `s != ROOT`)
2. `RIGHT-ARC` — `s` is the gold head of `b`
3. `REDUCE`    — `s` has a head **and** `b` still links to something deeper in the stack
4. `SHIFT`     — otherwise


In [ ]:
def _b_links_deeper_in_stack(config, b):
    """True if b has a gold arc (as head or dependent) to a word below the stack top."""
    for k in config.stack[:-1]:          # everything except the current top s
        if gold_head.get(b) == k:        # k is b's head
            return True
        if gold_head.get(k) == b:        # b is k's head
            return True
    return False

def oracle(config):
    "Return (action, label) for the current configuration."
    s, b = config.stack[-1], config.buffer[0]

    # 1. LEFT-ARC: b is head of s
    if gold_head.get(s) == b and s not in config.heads and s != 0:
        return ("LEFT-ARC", gold_label[s])

    # 2. RIGHT-ARC: s is head of b
    if gold_head.get(b) == s:
        return ("RIGHT-ARC", gold_label[b])

    # 3. REDUCE: s is finished and is blocking a deeper link for b
    if s in config.heads and _b_links_deeper_in_stack(config, b):
        return ("REDUCE", None)

    # 4. SHIFT
    return ("SHIFT", None)

print("Oracle defined.")

## 4 · Run the parser and print the trace

We loop until the buffer is empty, asking the oracle for each move, applying it, and
recording the configuration **before** the move (to match the README table).

In [ ]:
def apply(config, action, label):
    if   action == "SHIFT":     config.shift()
    elif action == "LEFT-ARC":  config.left_arc(label)
    elif action == "RIGHT-ARC": config.right_arc(label)
    elif action == "REDUCE":    config.reduce()
    else: raise ValueError(f"unknown action {action}")

def parse(tokens):
    config = Configuration(tokens)
    trace = []
    while not config.is_terminal():
        st, bf = config.snapshot()           # state BEFORE the move
        action, label = oracle(config)
        arc_before = len(config.arcs)
        apply(config, action, label)
        added = config.arcs[-1] if len(config.arcs) > arc_before else None
        trace.append((st, bf, action, label, added))
    return config, trace

def print_trace(trace):
    header = f"{'#':>2}  {'Stack':<26}{'Buffer':<32}{'Transition':<18}Arc added"
    print(header); print("-" * len(header))
    for i, (st, bf, action, label, added) in enumerate(trace):
        act = f"{action}({label})" if label else action
        if added:
            h, l, d = added
            arc = f"{tokens[h]} -> {tokens[d]} ({l})"
        else:
            arc = "-"
        print(f"{i:>2}  {st:<26}{bf:<32}{act:<18}{arc}")

final_config, trace = parse(tokens)
print_trace(trace)

## 5 · Verify the parse equals the gold tree

A correct oracle must reproduce **every** gold arc and **no** extras.

In [ ]:
produced = {(h, l, d) for (h, l, d) in final_config.arcs}
expected = {(h, l, d) for d, (h, l) in gold.items()}

print("Produced arcs:")
for h, l, d in sorted(produced, key=lambda x: x[2]):
    print(f"  {tokens[h]:>6} -> {tokens[d]:<6} ({l})")

print()
print("Matches gold tree:", produced == expected)
print("Transitions used :", len(trace), f"(<= 2n; n={len(tokens)-1} words)")
assert produced == expected, "Mismatch! Missing: %s  Extra: %s" % (expected - produced, produced - expected)

### Optional: pretty-print the tree

A tiny text renderer so you can *see* the structure (head above its dependents).

In [ ]:
from collections import defaultdict

children = defaultdict(list)
for h, l, d in final_config.arcs:
    children[h].append((d, l))

def render(node=0, depth=0):
    for d, l in sorted(children[node]):
        print("    " * depth + f"|- {tokens[d]} ({l})")
        render(d, depth + 1)

print("ROOT")
render()

## 6 · 🧩 Your turn

Parse **"She gave him the book ."** yourself, reusing everything above.

1. Fill in `gold2` — the dependency tree (`dependent -> (head, label)`).
   Hint: *gave* is the root; *She*=`nsubj`, *him*=`iobj`, *book*=`dobj`, *the*=`det` (of book), *.*=`punct`.
2. Point `gold`, `gold_head`, `gold_label`, `tokens` at the new sentence, then call `parse`.

Try it before revealing the solution cell below.


In [ ]:
# 🧩 Your turn — fill in the gold tree for "She gave him the book ."
tokens2 = ["ROOT", "She", "gave", "him", "the", "book", "."]

gold2 = {
    # dependent : (head, label)
    # 2: (0, "root"),
    # 1: (2, "nsubj"),
    # ... TODO: him (iobj), book (dobj), the (det of book), . (punct)
}

# Once gold2 is complete, uncomment to run:
# tokens = tokens2
# gold = gold2
# gold_head  = {d: h for d, (h, l) in gold.items()}
# gold_label = {d: l for d, (h, l) in gold.items()}
# cfg2, trace2 = parse(tokens)
# print_trace(trace2)

### ✅ Solution

In [ ]:
tokens = ["ROOT", "She", "gave", "him", "the", "book", "."]
gold = {
    2: (0, "root"),    # ROOT -> gave
    1: (2, "nsubj"),   # gave -> She
    3: (2, "iobj"),    # gave -> him
    5: (2, "dobj"),    # gave -> book
    4: (5, "det"),     # book -> the
    6: (2, "punct"),   # gave -> .
}
gold_head  = {d: h for d, (h, l) in gold.items()}
gold_label = {d: l for d, (h, l) in gold.items()}

cfg2, trace2 = parse(tokens)
print_trace(trace2)

produced2 = {(h, l, d) for (h, l, d) in cfg2.arcs}
expected2 = {(h, l, d) for d, (h, l) in gold.items()}
print("\nMatches gold tree:", produced2 == expected2)

## Recap

- A **configuration** `(stack, buffer, arcs)` + **four transitions** is all you need for
  arc-eager parsing.
- Each transition has a **precondition** (enforced here with `assert`).
- A **static oracle** turns a gold tree into the correct transition sequence — this is how
  you generate training data for a learned parser (replace the oracle with a classifier).
- The parser runs in **~2n steps / O(n)** and produces **projective** trees.

See [`README.md`](README.md) for the theory, the arc-eager vs arc-standard comparison,
and the exam cheat sheet.
